# 🎬 AI-Powered Movie Recommendation System

## Exploratory Data Analysis & Model Training Notebook

This notebook walks through the complete pipeline for building a **content-based movie
recommendation system** using the [TMDB 5000 Movie Dataset](https://www.kaggle.com/datasets/tmdb/tmdb-movie-metadata).

**Pipeline Overview:**
1. Load & merge TMDB datasets
2. Exploratory Data Analysis (EDA)
3. Data Preprocessing
4. Feature Engineering
5. Cosine Similarity Computation
6. Recommendation Queries
7. Model Persistence

---

## 1. Imports & Configuration

In [ ]:
import ast
import os
import pickle
import re
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import nltk
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings('ignore')
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Add src/ to path so we can import project modules
PROJECT_ROOT = Path('..').resolve()  # Notebook lives in notebooks/, root is one level up
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

print(f'Project root: {PROJECT_ROOT}')
print(f'Python version: {sys.version}')
print(f'Pandas version: {pd.__version__}')
print(f'NumPy version:  {np.__version__}')

## 2. Load Data

In [ ]:
from data_preprocessing import load_data

DATA_DIR = PROJECT_ROOT / 'data'
MOVIES_PATH  = DATA_DIR / 'tmdb_5000_movies.csv'
CREDITS_PATH = DATA_DIR / 'tmdb_5000_credits.csv'

# Load and merge both CSVs on 'title'
df_raw = load_data(MOVIES_PATH, CREDITS_PATH)

print(f'Merged dataset shape: {df_raw.shape}')
df_raw.head(3)

## 3. Exploratory Data Analysis (EDA)

### 3.1 Basic Statistics

In [ ]:
print('=== Dataset Info ===')
print(f'Rows:    {df_raw.shape[0]:,}')
print(f'Columns: {df_raw.shape[1]}')
print()
print('=== Numeric Summary ===')
display(df_raw[['budget', 'popularity', 'revenue', 'runtime', 'vote_average', 'vote_count']].describe().round(2))

In [ ]:
print('=== Missing Values ===')
null_counts = df_raw.isnull().sum()
null_pct    = (null_counts / len(df_raw) * 100).round(2)
null_df = pd.DataFrame({'null_count': null_counts, 'null_%': null_pct})
display(null_df[null_df['null_count'] > 0].sort_values('null_count', ascending=False))

### 3.2 Vote Average Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Vote average histogram
axes[0].hist(df_raw['vote_average'].dropna(), bins=30, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].set_title('Vote Average Distribution', fontsize=14)
axes[0].set_xlabel('Vote Average')
axes[0].set_ylabel('Count')
axes[0].axvline(df_raw['vote_average'].mean(), color='red', linestyle='--', label=f"Mean: {df_raw['vote_average'].mean():.2f}")
axes[0].legend()

# Popularity histogram (log scale)
axes[1].hist(df_raw['popularity'].dropna(), bins=50, color='coral', edgecolor='white', alpha=0.85)
axes[1].set_title('Popularity Distribution', fontsize=14)
axes[1].set_xlabel('Popularity')
axes[1].set_ylabel('Count')
axes[1].set_yscale('log')

plt.tight_layout()
plt.show()

## 4. Genre Distribution

In [ ]:
# Parse genres column to extract names
def _safe_parse(val):
    try:
        return ast.literal_eval(val)
    except Exception:
        return []

genres_parsed = df_raw['genres'].apply(_safe_parse)
all_genres = [item['name'] for sublist in genres_parsed for item in sublist if 'name' in item]

genre_counts = pd.Series(all_genres).value_counts().head(20)

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(genre_counts.index[::-1], genre_counts.values[::-1], color='mediumseagreen', edgecolor='white')
ax.set_title('Top 20 Movie Genres (TMDB 5000)', fontsize=15, fontweight='bold')
ax.set_xlabel('Number of Movies')

for bar, val in zip(bars, genre_counts.values[::-1]):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height() / 2,
            str(val), va='center', fontsize=9)

plt.tight_layout()
plt.show()
print(f'Total unique genres: {len(pd.Series(all_genres).unique())}')

## 5. Preprocessing Demo

In [ ]:
from data_preprocessing import preprocess_all

print('Running full preprocessing pipeline …')
df = preprocess_all(MOVIES_PATH, CREDITS_PATH)

print(f'\nShape after preprocessing: {df.shape}')
print(f'Columns: {list(df.columns)}')
df[['title', 'genres', 'keywords', 'cast', 'director']].head(5)

In [ ]:
# Inspect a single movie to verify parsing
sample = df[df['title'] == 'Avatar'].iloc[0]
print('Title    :', sample['title'])
print('Genres   :', sample['genres'])
print('Keywords :', sample['keywords'][:5], '…')
print('Cast     :', sample['cast'])
print('Director :', sample['director'])
print('Overview :', sample['overview'][:120], '…')

## 6. Feature Engineering

In [ ]:
from feature_engineering import build_feature_matrix

print('Building feature matrix …')
df_features, vectors, vectorizer = build_feature_matrix(df)

print(f'Feature matrix shape : {vectors.shape}')
print(f'Vocabulary size      : {len(vectorizer.vocabulary_):,}')
print()
print('Sample tags for "Avatar":')
avatar_tags = df_features[df_features['title'] == 'Avatar']['tags'].values
if len(avatar_tags):
    print(avatar_tags[0][:300], '…')

In [ ]:
# Visualise sparsity of the feature matrix
density = vectors.nnz / (vectors.shape[0] * vectors.shape[1])
print(f'Matrix sparsity: {(1 - density) * 100:.2f}%  |  Density: {density * 100:.2f}%')

# Top tokens in vocabulary
top_tokens = sorted(vectorizer.vocabulary_.items(), key=lambda x: x[1])[:20]
print('\nSample vocabulary tokens (first 20 by index):')
print([t for t, _ in top_tokens])

## 7. Cosine Similarity Computation

In [ ]:
from recommendation_engine import compute_similarity

print('Computing cosine similarity matrix …')
similarity = compute_similarity(vectors)

print(f'Similarity matrix shape : {similarity.shape}')
print(f'Matrix dtype            : {similarity.dtype}')
print(f'Min similarity          : {similarity.min():.4f}')
print(f'Max similarity          : {similarity.max():.4f}')
print(f'Mean off-diagonal sim   : {similarity[similarity < 1].mean():.4f}')

In [ ]:
# Visualise a small sub-matrix heatmap (first 50 movies)
import matplotlib.colors as mcolors

sub = similarity[:50, :50]
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(sub, cmap='YlOrRd', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label='Cosine Similarity')
ax.set_title('Cosine Similarity Heatmap (First 50 Movies)', fontsize=13)
ax.set_xlabel('Movie Index')
ax.set_ylabel('Movie Index')
plt.tight_layout()
plt.show()

## 8. Test Recommendations for 3 Movies

In [ ]:
from recommendation_engine import get_recommendations

test_movies = ['Avatar', 'The Dark Knight', 'The Godfather']

for movie in test_movies:
    print(f"\n{'='*55}")
    print(f"  Top 10 recommendations for: {movie}")
    print(f"{'='*55}")
    
    recs = get_recommendations(movie, df_features, similarity, top_n=10)
    
    if not recs:
        print('  Movie not found in dataset.')
    else:
        for rank, rec in enumerate(recs, 1):
            print(f"  {rank:2d}. {rec['title']:<45} [score: {rec['similarity_score']:.4f}]")

In [ ]:
# Visualise similarity scores for Avatar recommendations
avatar_recs = get_recommendations('Avatar', df_features, similarity, top_n=10)

if avatar_recs:
    titles = [r['title'] for r in avatar_recs]
    scores = [r['similarity_score'] for r in avatar_recs]

    fig, ax = plt.subplots(figsize=(10, 6))
    colors = plt.cm.Blues(np.linspace(0.4, 0.9, len(scores)))
    bars = ax.barh(titles[::-1], scores[::-1], color=colors[::-1], edgecolor='white')

    for bar, score in zip(bars, scores[::-1]):
        ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height() / 2,
                f'{score:.4f}', va='center', fontsize=9)

    ax.set_xlabel('Cosine Similarity Score', fontsize=12)
    ax.set_title('Top 10 Movies Similar to "Avatar"', fontsize=14, fontweight='bold')
    ax.set_xlim(0, max(scores) * 1.15)
    plt.tight_layout()
    plt.show()

## 9. Additional EDA — Release Year Trend

In [ ]:
from utils import get_release_year

df_features['release_year'] = df_features['release_date'].apply(get_release_year)
year_counts = df_features['release_year'].dropna().astype(int).value_counts().sort_index()

fig, ax = plt.subplots(figsize=(14, 5))
ax.fill_between(year_counts.index, year_counts.values, alpha=0.4, color='mediumpurple')
ax.plot(year_counts.index, year_counts.values, color='mediumpurple', linewidth=2)
ax.set_title('Number of Movies Released per Year (TMDB 5000)', fontsize=14)
ax.set_xlabel('Year')
ax.set_ylabel('Movie Count')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Save Model Artefacts

In [ ]:
from recommendation_engine import save_model, load_model

MODEL_DIR = PROJECT_ROOT / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(f'Saving model artefacts to: {MODEL_DIR}')
save_model(similarity, df_features, path=MODEL_DIR)

sim_size_mb = (MODEL_DIR / 'similarity.pkl').stat().st_size / 1024 / 1024
mov_size_mb = (MODEL_DIR / 'movies.pkl').stat().st_size / 1024 / 1024

print(f'  similarity.pkl : {sim_size_mb:.2f} MB')
print(f'  movies.pkl     : {mov_size_mb:.2f} MB')
print('Model saved successfully!')

In [ ]:
# Verify the saved model by loading it back
print('Loading model from disk to verify …')
similarity_loaded, movies_loaded = load_model(path=MODEL_DIR)

print(f'Loaded similarity shape: {similarity_loaded.shape}')
print(f'Loaded movies shape    : {movies_loaded.shape}')

# Quick sanity-check: recommendations should be identical
recs_original = get_recommendations('Avatar', df_features, similarity, top_n=5)
recs_loaded   = get_recommendations('Avatar', movies_loaded, similarity_loaded, top_n=5)

assert [r['title'] for r in recs_original] == [r['title'] for r in recs_loaded], \
    'Loaded model gives different results!'

print('\n✅ Model verified — loaded artefacts produce identical recommendations.')
print('\nTop 5 for "Avatar" (from loaded model):')
for i, rec in enumerate(recs_loaded, 1):
    print(f'  {i}. {rec["title"]}')

---

## Summary

| Step | Description | Output |
|------|-------------|--------|
| 1 | Load & merge CSVs | 4,803 × 23 DataFrame |
| 2 | Clean & parse | Structured lists per column |
| 3 | Feature engineering | Tags string + (4803 × 5000) sparse matrix |
| 4 | Cosine similarity | (4803 × 4803) float64 matrix |
| 5 | Recommendations | Top-N list of dicts |
| 6 | Persist | `similarity.pkl` + `movies.pkl` |

The model is ready to be loaded by the **Streamlit web app** (`app.py`) for
real-time recommendations.